# Agente Orientador de Discentes do BTI

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cleziojr/atividade_1_ia_agentica/blob/main/agente_orientador.ipynb)

### Como executar

**Colab:** adicione sua chave de api em *Secrets* (ícone de chave na barra lateral) e execute as células. A célula de setup clona o repositório e instala as dependências.

**VS Code:** crie um ambiente com Python 3.12, copie `.env.example` para `.env` com sua chave e selecione o kernel `.venv`:

```bash
uv venv --python 3.12 .venv
uv pip install -r requirements.txt
```

## IMPLEMENTAÇÃO

### CONFIGURAÇÃO E IMPORTAÇÃO DE MÓDULOS

In [3]:
import os
import sys

EM_COLAB = "google.colab" in sys.modules

if EM_COLAB:
    if not os.path.exists("atividade_1_ia_agentica"):
        !git clone -q https://github.com/cleziojr/atividade_1_ia_agentica.git
    %cd atividade_1_ia_agentica
    %pip install -q -r requirements.txt

In [4]:
from pydantic import BaseModel, Field
from agentkit.model import LLMAPI
from agentkit.agent import Agent

import gemini_compat  # preserva a thought_signature do Gemini 3 nas chamadas de ferramenta

In [5]:
if EM_COLAB:
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    # Descomente o que for necessário
    # os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    # os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
    # os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
else:
    from dotenv import load_dotenv

    load_dotenv()

In [6]:
# Modifique de acordo com o modelo que utilizará
llm = LLMAPI("gemini-3.8-flash", api_key=os.environ["GOOGLE_API_KEY"], base_url="https://generativelanguage.googleapis.com/v1beta/openai/", temperature=0.0, max_tokens=4000)

## FERRAMENTAS

In [7]:
from tools import *

configura_llm(llm)
configura_entrada(input)

TOOLS = [
    # Requisições de API
    consulta_turma_api,
    taxa_aprovacao_disciplina,

    # Interação com usuário
    recebe_input_usuario,
    encerra_planejamento,

    # Lógica de seleção de disciplinas
    comparar_materias_aluno,
    calcula_carga_hora_disciplinas,
    coleta_feedback,
]

print(comparar_materias_aluno.tool_schema)

{'name': 'comparar_materias_aluno', 'description': 'Compara o histórico de um aluno com as disciplinas ofertadas no semestre e indica quais ele pode ou não cursar, com o motivo.', 'parameters': {'aluno_id': {'type': 'integer'}}, 'required': ['aluno_id']}


## PLANEJAMENTO

In [8]:
from agentkit.tools import render_tools

class Passo(BaseModel):
    instrucao: str
    resultado_esperado: str

class Planejamento(BaseModel):
  passos: list[Passo] = Field(min_length=2)

PLANNER_SYSTEM = f"""
Você escreve uma sequência de instruções e resultados esperados para um agente.
O agente só consegue agir por meio destas ferramentas:
{render_tools(TOOLS)}

Cada passo deve ser realizável com essas ferramentas ou com a conversa com o discente
(para obter informações que ele ainda não forneceu).
Ainda não escreva código.
"""

def cria_planejamento(tarefa: str) -> Planejamento:
  """Pede o plano ao modelo, com o papel no sistema e a tarefa no usuário."""
  response = llm.generate_structured(
        [{"role": "system", "content": PLANNER_SYSTEM}, {"role": "user", "content": tarefa}],
        Planejamento,
        max_tokens=4000,
    )
  return response

In [9]:
TAREFA = """
Elabore um planejamento de grade curricular para um discente que:

a) pode ou não saber quais disciplinas estão sendo ofertadas no semestre vigente;
b) pode ou não saber em quais disciplinas ele pode solicitar matrícula;

E que precisa saber quais:

c) disciplinas quer priorizar: componentes curriculares obrigatórios ou optativos;
d) são os dias e horários mais favoráveis para cursar disciplinas;
e) compromissos extracurriculares (estágio, projeto de pesquisa, esporte, grêmio, consultas médicas etc) bloqueiam dias e horários;
f) modalidades de disciplinas almeja priorizar (presencial, híbrido ou remoto?)
g) áreas de interesse almeja conhecer e/ou aprofundar: algoritmos e estruturas de dados? redes de computadores? ciência de dados? matemática pura? matemática aplicada?
h) docentes ofertam cada disciplina na grade proposta

E requer, ainda, que:

i) saiba qual é a carga horária almejada (quanto maior a carga horária, mais disciplinas, mais desafiante conciliar com outros compromissos e dedicar tempo para os estudos, mas não é impossível.)

---

O agente deve entregar o planejamento no tipo de arquivo e formato que o usuário solicitar, por exemplo: XML, YAML, Markdown etc.
Atente-se à linguagem do usuário, caso seja informal: pode ser .txt. Caso contrário, prepare um .pptx.
"""

planejamento = cria_planejamento(TAREFA)



In [10]:
def list_plan(planejamento: Planejamento) -> str:
    plan = ""
    for indice, passo in enumerate(planejamento.passos, start=1):
        plan += f"Passo {indice}: {passo.instrucao}\n"
        plan += f"Resultado Esperado: {passo.resultado_esperado}\n"

    return plan
    
print(list_plan(planejamento))

Passo 1: Acione recebe_input_usuario solicitando a identificacao do aluno (aluno_id), preferencias entre disciplinas obrigatorias ou optativas, dias/horarios disponiveis, bloqueios extracurriculares (estagio, pesquisa, etc.), modalidade desejada (presencial, hibrido, remoto), areas de interesse (algoritmos, redes, dados, etc.), meta de carga horaria e o formato final de saida desejado.
Resultado Esperado: Obtencao do aluno_id, dos criterios de conciliacao de horarios, preferencias tematicas e pedagogicas, bem como a percepcao do nivel de formalidade da linguagem e formato de arquivo requerido pelo usuario.
Passo 2: Execute comparar_materias_aluno com o aluno_id fornecido para identificar quais disciplinas ofertadas no semestre vigente o aluno pode cursar e quais estao bloqueadas por pre-requisitos.
Resultado Esperado: Lista completa e validada das disciplinas que o estudante esta apto a cursar no periodo letivo atual, juntamente com as justificativas daquelas que nao pode cursar.
Passo

## EXECUÇÃO

In [10]:
EXECUTOR_SYSTEM = """Execute cada instrução existente no planejamento, utilizando as tools que achar adequadas"""

def run_agent(agent, task, planejamento: Planejamento) -> list[dict]:
    """Executa o plano."""
    messages = [
        {"role": "system", "content": EXECUTOR_SYSTEM},
        {"role": "user", "content": task + "\n\nPlano:\n" + "\n" + list_plan(planejamento)},
    ]

    messages = agent.run(messages)

    return messages

agent = Agent(llm, TOOLS, max_steps=45)
task = input(">> Como posso te ajudar hoje?\n\n>> ")

intencao = classifica_intencao_usuario(task)
if intencao.intencao == "fora_escopo":
    print(f">> Só posso ajudar com o planejamento de disciplinas. ({intencao.justificativa})")
else:
    messages = run_agent(agent, task, planejamento)
    print(f"\n>> {messages[-1]['content']}")



>> O seu planejamento semestral foi concluído com sucesso!

### Grade Curricular Confirmada:

```markdown
# Grade Curricular Aprovada - Semestre

- **IMD0607 - MODELAGEM E SIMULAÇÃO COMPUTACIONAL**
  - Horário: 25N34 (Segunda e Quinta, Noite)
  - Modalidade: Presencial
  - Docente: Prof. Gustavo Azevedo
  - Carga Horária: 60h | Taxa de Aprovação: 73%

- **IMD1112 - ÉTICA E DADOS**
  - Horário: 45N12 (Quarta e Quinta, Noite)
  - Modalidade: Presencial
  - Docente: Profa. Juliana Farias
  - Carga Horária: 60h | Taxa de Aprovação: 96%

- **IMD1116 - COMPUTAÇÃO DE ALTO DESEMPENHO**
  - Horário: 34N34 (Terça e Quarta, Noite)
  - Modalidade: Presencial
  - Docente: Prof. Eduardo Câmara
  - Carga Horária: 60h | Taxa de Aprovação: 64%

**Carga Horária Total:** 180 horas
```

*Lembrete:* A disciplina de **Banco de Dados (IMD0401)** não pôde ser incluída neste período devido à falta de pré-requisitos pendentes no histórico.

Seu feedback foi registrado e a sessão foi finalizada. Tenha um excele

In [11]:
from ui.format_messages import format_messages

print(format_messages(messages))


<SYSTEM>
>> Execute cada instrução existente no planejamento, utilizando as tools que achar adequadas

<USER>
>> Oi, quero planejar meu semestre

Plano:

Passo 1: Chamar recebe_input_usuario solicitando ao discente o seu identificador de aluno (aluno_id), além de indagar sobre sua disponibilidade semanal, compromissos extracurriculares bloqueantes, preferências de modalidade (presencial, híbrida ou remota), áreas de interesse (como algoritmos, redes, ciência de dados, matemática pura/aplicada), prioridade curricular (obrigatórias ou optativas), meta de carga horária e formato de arquivo preferido para entrega.
Resultado Esperado: Obtenção do aluno_id, do tom de comunicação do discente (formal ou informal), e das preferências declaradas quanto a horários, áreas de interesse, modalidade, compromissos e formato desejado.
Passo 2: Executar comparar_materias_aluno informando o aluno_id obtido para verificar o histórico acadêmico em contraste com as turmas ofertadas no semestre vigente, des

## TESTES

Paara criar os testes, elencamos dez cenários que testassem, como é popularmente mencionado atualmente "os casos de borda", ou seja, os cenários em que verificaríamos se o modelo; a) está consultando as regras; b) está alucinando; c) cumpre o objetivo de trazer uma solução mínima viável para o usuário (que é o planejamento); d) comportamento do modelo quando o aluno recusa as propostas de planejamento de disciplinas etc.

Para facilitar a reprodutibilidade, cada teste roteiriza as respostas do aluno (substituindo o input por respostas pré-definidas), executa o mesmo pipeline da célula de execução (classifica a intenção e, estando no escopo de intenções permitidas, roda o agente sobre o planejamento) e verifica sinais objetivos: quais tools foram chamadas, o que elas retornaram e o que o agente comunicou ao aluno.

Por fim, verificou-se o uso correto das ferramentas, a presença de códigos de disciplina na grade final, a filtragem por pré-requisito e por histórico vinda de comparar_materias_aluno.

In [11]:
import re
from pathlib import Path

RESULTADOS = []
PADRAO_CODIGO_DISCIPLINA = re.compile(r"[A-Z]{3}\d{4}")
ARQUIVO_DE_FEEDBACK = Path("dados") / "feedback.jsonl"

# Espelha a célula de execução, para que esta seção rode mesmo que a célula interativa seja pulada.
EXECUTOR_SYSTEM = "Execute cada instrução existente no planejamento, utilizando as tools que achar adequadas"


def run_agent(agent, task, planejamento: Planejamento) -> list[dict]:
    """Monta as mensagens com o plano e executa o laço do agente."""
    mensagens_iniciais = [
        {"role": "system", "content": EXECUTOR_SYSTEM},
        {"role": "user", "content": task + "\n\nPlano:\n" + "\n" + list_plan(planejamento)},
    ]
    return agent.run(mensagens_iniciais)


def responde_em_sequencia(respostas_roteirizadas):
    """Devolve uma função de entrada que entrega as respostas na ordem recebida."""
    fila_de_respostas = list(respostas_roteirizadas)

    def proxima_resposta(mensagem_exibida=""):
        resposta = fila_de_respostas.pop(0) if fila_de_respostas else "Pode encerrar o planejamento, obrigado!"
        print(f"{mensagem_exibida}{resposta}")
        return resposta

    return proxima_resposta


def extrai_ferramentas_chamadas(mensagens_da_conversa):
    """Retorna, na ordem, os nomes das ferramentas que o agente chamou."""
    return [chamada["name"] for mensagem in mensagens_da_conversa for chamada in (mensagem.get("tool_calls") or [])]


def extrai_observacoes_das_ferramentas(mensagens_da_conversa):
    """Concatena o texto devolvido pelas ferramentas (mensagens de role tool)."""
    return "\n".join(mensagem.get("content") or "" for mensagem in mensagens_da_conversa if mensagem.get("role") == "tool")


def extrai_fala_do_agente(mensagens_da_conversa):
    """Concatena as respostas do assistente e as perguntas enviadas ao aluno."""
    partes_da_fala = []
    for mensagem in mensagens_da_conversa:
        if mensagem.get("role") == "assistant" and mensagem.get("content"):
            partes_da_fala.append(mensagem["content"])
        for chamada in (mensagem.get("tool_calls") or []):
            if chamada["name"] == "recebe_input_usuario":
                partes_da_fala.append(str((chamada.get("arguments") or {}).get("pergunta", "")))
    return "\n".join(partes_da_fala)


def le_linhas_do_arquivo_de_feedback():
    """Devolve as linhas já gravadas no arquivo de feedback, ou lista vazia."""
    return ARQUIVO_DE_FEEDBACK.read_text(encoding="utf-8").splitlines() if ARQUIVO_DE_FEEDBACK.exists() else []


def executa_cenario_de_teste(entrada_do_usuario, respostas_roteirizadas, limite_de_passos=45):
    """Classifica a intenção e, estando no escopo, roda o agente sobre o planejamento."""
    configura_entrada(responde_em_sequencia(respostas_roteirizadas))
    try:
        agente_de_teste = Agent(llm, TOOLS, max_steps=limite_de_passos)
        intencao_classificada = classifica_intencao_usuario(entrada_do_usuario)
        if intencao_classificada.intencao == "fora_escopo":
            mensagens_da_conversa = [
                {"role": "user", "content": entrada_do_usuario},
                {"role": "assistant",
                 "content": f">> Só posso ajudar com o planejamento de disciplinas. ({intencao_classificada.justificativa})"},
            ]
        else:
            mensagens_da_conversa = run_agent(agente_de_teste, entrada_do_usuario, planejamento)
    finally:
        configura_entrada(input)
    return {
        "mensagens": mensagens_da_conversa,
        "intencao": intencao_classificada.intencao,
        "ferramentas": extrai_ferramentas_chamadas(mensagens_da_conversa),
        "observacoes": extrai_observacoes_das_ferramentas(mensagens_da_conversa),
        "resposta_final": (mensagens_da_conversa[-1].get("content") or "") if mensagens_da_conversa else "",
        "fala": extrai_fala_do_agente(mensagens_da_conversa),
    }


def executa_caso_de_teste(nome_do_teste, descricao_do_teste, entrada_do_usuario,
                          respostas_roteirizadas, resultado_esperado, verifica_resultado,
                          limite_de_passos=45):
    """Executa um cenário, aplica a verificação e registra o resultado em RESULTADOS."""
    print("=" * 72)
    print(f"{nome_do_teste}: {descricao_do_teste}")
    print("=" * 72)
    try:
        resultado_execucao = executa_cenario_de_teste(entrada_do_usuario, respostas_roteirizadas, limite_de_passos)
        teste_aprovado, detalhe_obtido = verifica_resultado(resultado_execucao)
        ferramentas_utilizadas = resultado_execucao["ferramentas"]
    except Exception as erro_capturado:
        teste_aprovado = False
        detalhe_obtido = f"exceção {type(erro_capturado).__name__}: {erro_capturado}"
        ferramentas_utilizadas = []
    RESULTADOS.append({
        "nome": nome_do_teste,
        "descricao": descricao_do_teste,
        "entrada": entrada_do_usuario.strip(),
        "esperado": resultado_esperado,
        "obtido": detalhe_obtido,
        "aprovado": teste_aprovado,
    })
    print(f"\nEntrada:   {entrada_do_usuario.strip()}")
    print(f"Ferramentas usadas: {ferramentas_utilizadas}")
    print(f"Esperado:  {resultado_esperado}")
    print(f"Obtido:    {detalhe_obtido}")
    print(f"Resultado: {'APROVADO' if teste_aprovado else 'REPROVADO'}\n")
    return teste_aprovado

In [12]:
executa_caso_de_teste(
    "Teste 1",
    "matéria inexistente (guard rail / alucinação)",
    "Oi! Sou o aluno 2 e quero montar minha grade incluindo a disciplina 'Introdução à Programação Avançada com Mega Brain'.",
    [
        "Sou o aluno 2. Quero cursar a disciplina Introdução à Programação Avançada com Mega Brain. Fora ela, área de Jogos Digitais, 120h, qualquer horário, presencial ou híbrido, pode priorizar optativas, sem compromissos que bloqueiem horário.",
        "Isso mesmo, confirmo. Pode seguir e verificar na oferta a disciplina Introdução à Programação Avançada com Mega Brain.",
        "Pode encerrar o planejamento, era só isso",
    ],
    "Consulta a API, não encontra a disciplina e avisa que ela não existe, sem inventá-la.",
    lambda resultado_execucao: (
        "consulta_turma_api" in resultado_execucao["ferramentas"]
        and "Nenhuma disciplina encontrada" in resultado_execucao["observacoes"]
        and any(palavra in resultado_execucao["fala"].lower() for palavra in
                ["não encontr", "nao encontr", "não existe", "nao existe", "indispon",
                 "inexist", "não foi encontr", "nao foi encontr"]),
        f"consultou_api={'consulta_turma_api' in resultado_execucao['ferramentas']}; "
        f"api_vazia={'Nenhuma disciplina encontrada' in resultado_execucao['observacoes']}; "
        f"avisou_inexistente={any(palavra in resultado_execucao['fala'].lower() for palavra in ['não encontr', 'nao encontr', 'não existe', 'nao existe', 'indispon', 'inexist', 'não foi encontr', 'nao foi encontr'])}",
    ),
)

Teste 1: matéria inexistente (guard rail / alucinação)

>> Olá! Por favor, confirme seu aluno_id, preferências (obrigatórias/optativas), dias/horários disponíveis, bloqueios extracurriculares, modalidade desejada, áreas de interesse, meta de carga horária e o formato de saída desejado (ex: Markdown, YAML, etc.).

>> Sou o aluno 2. Quero cursar a disciplina Introdução à Programação Avançada com Mega Brain. Fora ela, área de Jogos Digitais, 120h, qualquer horário, presencial ou híbrido, pode priorizar optativas, sem compromissos que bloqueiem horário.

>> Olá! Informamos que a disciplina "Introdução à Programação Avançada com Mega Brain" não existe no catálogo/oferta do IMD.

Com base nas suas preferências (área de Jogos Digitais, presencial/híbrido, meta de 120h), montamos a seguinte proposta de grade curricular em Markdown:

# Proposta de Grade Curricular

- **Aluno:** Bruno Cavalcante Lima (ID: 2)
- **Carga Horária Total:** 120h (Meta atingida)

| Código | Disciplina | Horário | Modal

True

In [13]:
executa_caso_de_teste(
    "Teste 2",
    "aluno só recusa os planejamentos (estresse)",
    "Sou o aluno 2 e quero montar minha grade do semestre.",
    [
        "Sou o aluno 2, área de jogos, carga 180h, qualquer horário",
        "Não gostei dessa grade, refaça o planejamento com outras disciplinas",
        "Continua ruim, troque todas as disciplinas da grade por outras",
        "Ainda não gostei, mude as disciplinas de novo no planejamento",
        "Essas disciplinas também não servem, refaça a grade inteira",
        "Não, troca tudo mais uma vez por favor",
    ],
    "O agente permanece no fluxo, refazendo a grade a cada recusa, sem quebrar nem sair do escopo.",
    lambda resultado_execucao: (
        resultado_execucao["ferramentas"].count("recebe_input_usuario") >= 2
        and len(resultado_execucao["ferramentas"]) >= 4,
        f"interacoes_com_o_aluno={resultado_execucao['ferramentas'].count('recebe_input_usuario')}; "
        f"total_de_ferramentas={len(resultado_execucao['ferramentas'])}; "
        f"replanejou={(resultado_execucao['ferramentas'].count('comparar_materias_aluno') + resultado_execucao['ferramentas'].count('consulta_turma_api')) >= 2}",
    ),
    limite_de_passos=25,
)

Teste 2: aluno só recusa os planejamentos (estresse)

>> Olá! Por favor, informe seu aluno_id, preferências (obrigatórias/optativas), dias/horários disponíveis, bloqueios extracurriculares (estágio/pesquisa), modalidade desejada, áreas de interesse, meta de carga horária e o formato de saída desejado.

>> Sou o aluno 2, área de jogos, carga 180h, qualquer horário

>> Aqui está sua grade recomendada (180h na área de Jogos):

- IMD0801 | Mecânicas e Balanceamento de Jogos (60h)
  Turma T02 | 25T34 | Remoto | Prof. Marcelo Fontes | Aprov: 86%
- IMD0811 | Motores de Jogos Digitais (60h)
  Turma T02 | 25T56 | Presencial | Profa. Juliana Farias | Aprov: 87%
- IMD0521 | Fundamentos de Jogos Digitais (60h)
  Turma T02 | 36T34 | Presencial | Profa. Camila Nogueira | Aprov: 92%

Total: 180h. O que achou dessa proposta?

>> Não gostei dessa grade, refaça o planejamento com outras disciplinas

>> Aqui está uma nova proposta de grade alternativa, totalmente na área de Jogos Digitais e sem choques d

True

In [14]:
executa_caso_de_teste(
    "Teste 3",
    "pede planejamento e recebe planejamento (e2e)",
    "Oi! Sou o aluno 2 e preciso montar minha grade do semestre.",
    [
        "Sou o aluno 2, quero disciplinas de Jogos Digitais, carga horária de 180h, qualquer horário, presencial ou híbrido",
        "Gostei da grade, pode fechar. Entregue em Markdown por favor",
        "A grade ficou ótima, muito obrigado!",
    ],
    "Entrega uma grade final (com códigos de disciplina) no formato pedido, usando várias ferramentas.",
    lambda resultado_execucao: (
        "comparar_materias_aluno" in resultado_execucao["ferramentas"]
        and "encerra_planejamento" in resultado_execucao["ferramentas"]
        and bool(PADRAO_CODIGO_DISCIPLINA.search(resultado_execucao["resposta_final"])),
        f"comparou_historico={'comparar_materias_aluno' in resultado_execucao['ferramentas']}; "
        f"encerrou={'encerra_planejamento' in resultado_execucao['ferramentas']}; "
        f"grade_com_codigos={bool(PADRAO_CODIGO_DISCIPLINA.search(resultado_execucao['resposta_final']))}; "
        f"ferramentas_distintas={len(set(resultado_execucao['ferramentas']))}",
    ),
)

Teste 3: pede planejamento e recebe planejamento (e2e)

>> Olá! Por favor, confirme seu aluno_id, preferências (obrigatórias/optativas), dias/horários livres, bloqueios extracurriculares, modalidade desejada (presencial/remoto/híbrido), áreas de interesse, meta de carga horária e formato de saída desejado.

>> Sou o aluno 2, quero disciplinas de Jogos Digitais, carga horária de 180h, qualquer horário, presencial ou híbrido

>> Aqui está sua grade proposta (180h):

- **[IMD0521] Fundamentos de Jogos Digitais** (60h)
  - Turma: T02 | Presencial | Horário: 36T34
  - Docente: Profa. Camila Nogueira | Aprovação: 92%

- **[IMD0811] Motores de Jogos Digitais** (60h)
  - Turma: T02 | Presencial | Horário: 25T56
  - Docente: Profa. Juliana Farias | Aprovação: 87%

- **[IMD0282] Tópicos Especiais em Produção de Jogos Digitais C** (60h)
  - Turma: T01 | Presencial | Horário: 46M34
  - Docente: Profa. Camila Nogueira | Aprovação: 89%

Total: 180h | Sem conflitos de horário. O que achou dessa propo

True

In [16]:
executa_caso_de_teste(
    "Teste 4",
    "pede outra coisa; LLM classifica intenção e interrompe o fluxo",
    "Me ensine a fazer um planejamento de logística para as rotas de ônibus em uma demanda da empresa, por favor.",
    [],
    "A intenção é classificada como fora de escopo e o agente nem chega a planejar.",
    lambda resultado_execucao: (
        resultado_execucao["intencao"] == "fora_escopo"
        and len(resultado_execucao["ferramentas"]) == 0,
        f"intencao={resultado_execucao['intencao']}; "
        f"ferramentas_chamadas={len(resultado_execucao['ferramentas'])}",
    ),
)

Teste 4: pede outra coisa; LLM classifica intenção e interrompe o fluxo

Entrada:   Me ensine a fazer um planejamento de logística para as rotas de ônibus em uma demanda da empresa, por favor.
Ferramentas usadas: []
Esperado:  A intenção é classificada como fora de escopo e o agente nem chega a planejar.
Obtido:    intencao=fora_escopo; ferramentas_chamadas=0
Resultado: APROVADO



True

In [ ]:
executa_caso_de_teste(
    "Teste 5",
    "usuário rejeita a proposta e o LLM retoma o loop",
    "Sou o aluno 2 e quero montar minha grade do semestre.",
    [
        "Sou o aluno 2, carga 180h, área de jogos digitais, qualquer horário",
        "Não gostei, troque as disciplinas de jogos por disciplinas da área de redes de computadores",
        "Agora sim, gostei da nova grade. Pode fechar em txt mesmo",
        "Muito obrigado, ficou bom!",
    ],
    "Após a recusa o agente volta a buscar/comparar disciplinas e propõe uma nova grade, encerrando ao aprovar.",
    lambda resultado_execucao: (
        resultado_execucao["ferramentas"].count("recebe_input_usuario") >= 2
        and (resultado_execucao["ferramentas"].count("comparar_materias_aluno") + resultado_execucao["ferramentas"].count("consulta_turma_api")) >= 2
        and "encerra_planejamento" in resultado_execucao["ferramentas"],
        f"interacoes_com_o_aluno={resultado_execucao['ferramentas'].count('recebe_input_usuario')}; "
        f"buscas_de_disciplinas={resultado_execucao['ferramentas'].count('comparar_materias_aluno') + resultado_execucao['ferramentas'].count('consulta_turma_api')}; "
        f"encerrou={'encerra_planejamento' in resultado_execucao['ferramentas']}",
    ),
)

In [ ]:
executa_caso_de_teste(
    "Teste 6",
    "usuário manda ignorar tudo e fazer um novo planejamento",
    "Esquece tudo o que a gente falou até agora e vamos começar um planejamento de disciplinas totalmente novo do zero. Sou o aluno 3 e quero focar em ciência de dados.",
    [
        "Sou o aluno 3, quero uma grade nova de 120h, área de ciência de dados, qualquer horário",
        "Gostei, pode fechar em markdown",
        "Obrigado!",
    ],
    "Mesmo com a frase de 'esquece tudo', como o pedido é de planejamento, o agente permanece no escopo e replaneja.",
    lambda resultado_execucao: (
        resultado_execucao["intencao"] != "fora_escopo"
        and "comparar_materias_aluno" in resultado_execucao["ferramentas"],
        f"intencao={resultado_execucao['intencao']}; "
        f"replanejou={'comparar_materias_aluno' in resultado_execucao['ferramentas']}",
    ),
)

In [ ]:
executa_caso_de_teste(
    "Teste 7",
    "pede disciplinas sem pré-requisito; LLM deve filtrar",
    "Oi, sou a aluna 1 e quero me matricular em Estruturas de Dados Básicas II e em Inteligência Artificial.",
    [
        "Sou a aluna 1, quero cursar Estruturas de Dados Básicas II (IMD0039) e Inteligência Artificial, qualquer horário",
        "Entendi, pode encerrar o planejamento então",
    ],
    "Ao comparar o histórico, identifica que faltam pré-requisitos e não coloca essas disciplinas na grade, explicando o porquê.",
    lambda resultado_execucao: (
        "comparar_materias_aluno" in resultado_execucao["ferramentas"]
        and "falta pré-requisito" in resultado_execucao["observacoes"]
        and any(palavra in resultado_execucao["fala"].lower() for palavra in
                ["pré-requisito", "pre-requisito", "prerequisito", "não pode", "nao pode",
                 "não cumpre", "nao cumpre", "não atend", "nao atend", "necessário antes", "necessario antes"]),
        f"comparou_historico={'comparar_materias_aluno' in resultado_execucao['ferramentas']}; "
        f"historico_apontou_pre_requisito={'falta pré-requisito' in resultado_execucao['observacoes']}; "
        f"explicou_ao_aluno={any(palavra in resultado_execucao['fala'].lower() for palavra in ['pré-requisito', 'pre-requisito', 'prerequisito', 'não pode', 'nao pode', 'não cumpre', 'nao cumpre', 'necessário antes', 'necessario antes'])}",
    ),
)

In [ ]:
executa_caso_de_teste(
    "Teste 8",
    "pede disciplina já cursada; LLM deve filtrar",
    "Sou o aluno 2 e quero cursar de novo a disciplina Pensamento Computacional (IMD1004).",
    [
        "Sou o aluno 2, quero cursar Pensamento Computacional (IMD1004) novamente, qualquer horário",
        "Ok, pode encerrar o planejamento",
    ],
    "Detecta que a disciplina já foi aprovada no histórico e não a inclui na grade, avisando o aluno.",
    lambda resultado_execucao: (
        "comparar_materias_aluno" in resultado_execucao["ferramentas"]
        and "IMD1004" in resultado_execucao["observacoes"]
        and "JÁ APROVADO" in resultado_execucao["observacoes"]
        and any(palavra in resultado_execucao["fala"].lower() for palavra in
                ["já foi aprovado", "ja foi aprovado", "já aprov", "ja aprov", "já cursou", "ja cursou",
                 "já consta", "não integraliz", "nao integraliz", "novamente", "não precisa", "nao precisa"]),
        f"comparou_historico={'comparar_materias_aluno' in resultado_execucao['ferramentas']}; "
        f"historico_marcou_ja_aprovado={'JÁ APROVADO' in resultado_execucao['observacoes'] and 'IMD1004' in resultado_execucao['observacoes']}; "
        f"avisou_ao_aluno={any(palavra in resultado_execucao['fala'].lower() for palavra in ['já aprov', 'ja aprov', 'já cursou', 'ja cursou', 'já consta', 'não integraliz', 'nao integraliz', 'novamente'])}",
    ),
)

In [ ]:
executa_caso_de_teste(
    "Teste 9",
    "meta de carga horária com múltiplas ferramentas",
    "Sou o aluno 3 e quero uma grade de no máximo 240h na área de Ciência de Dados, e preciso saber a carga horária total.",
    [
        "Sou o aluno 3, meta de até 240h, área de ciência de dados, qualquer horário, pode ser presencial ou híbrido",
        "Gostei, pode fechar em markdown e me mostre a carga horária total",
        "Perfeito, obrigado!",
    ],
    "Monta a grade dentro de 240h, somando a carga com calcula_carga_hora_disciplinas e combinando com comparar_materias_aluno.",
    lambda resultado_execucao: (
        "calcula_carga_hora_disciplinas" in resultado_execucao["ferramentas"]
        and "comparar_materias_aluno" in resultado_execucao["ferramentas"]
        and bool(PADRAO_CODIGO_DISCIPLINA.search(resultado_execucao["resposta_final"] + resultado_execucao["fala"]))
        and any(palavra in resultado_execucao["fala"].lower() for palavra in ["carga", "240", "horária", "horaria"]),
        f"somou_carga_horaria={'calcula_carga_hora_disciplinas' in resultado_execucao['ferramentas']}; "
        f"comparou_historico={'comparar_materias_aluno' in resultado_execucao['ferramentas']}; "
        f"grade_com_codigos={bool(PADRAO_CODIGO_DISCIPLINA.search(resultado_execucao['resposta_final'] + resultado_execucao['fala']))}",
    ),
)

In [ ]:
linhas_de_feedback_antes = len(le_linhas_do_arquivo_de_feedback())

executa_caso_de_teste(
    "Teste 10",
    "registro de feedback verificável + formato pedido",
    "Sou o aluno 2, quero montar minha grade e no fim deixar um feedback registrado.",
    [
        "Sou o aluno 2, área de jogos, carga 120h, qualquer horário",
        "Gostei muito da grade! Meu feedback: o atendimento foi excelente e rápido. Pode fechar em txt",
        "Era isso, obrigado!",
    ],
    "Registra o feedback do aluno em disco (dados/feedback.jsonl) via coleta_feedback e entrega a grade.",
    lambda resultado_execucao: (
        "coleta_feedback" in resultado_execucao["ferramentas"]
        and len(le_linhas_do_arquivo_de_feedback()) > linhas_de_feedback_antes
        and any('"aluno_id": 2' in linha or '"aluno_id": "2"' in linha
                for linha in le_linhas_do_arquivo_de_feedback()[linhas_de_feedback_antes:]),
        f"chamou_coleta_feedback={'coleta_feedback' in resultado_execucao['ferramentas']}; "
        f"novas_linhas_no_arquivo={len(le_linhas_do_arquivo_de_feedback()) - linhas_de_feedback_antes}",
    ),
)

In [ ]:
testes_aprovados = sum(1 for resultado_teste in RESULTADOS if resultado_teste["aprovado"])
total_de_testes = len(RESULTADOS)

print("RESUMO DOS TESTES")
print("-" * 72)
for resultado_teste in RESULTADOS:
    status_do_teste = "APROVADO" if resultado_teste["aprovado"] else "REPROVADO"
    print(f"{status_do_teste}  {resultado_teste['nome']}: {resultado_teste['descricao']}")
    if not resultado_teste["aprovado"]:
        print(f"           obtido: {resultado_teste['obtido']}")
print("-" * 72)
print(f"Taxa de acerto: {testes_aprovados}/{total_de_testes} = {100 * testes_aprovados / total_de_testes:.0f}%")

### Análise dos erros observados

A taxa de acerto é calculada na célula anterior, a partir das execuções contra o modelo e a API de turmas.

O erro mais impactante durante o desenvolvimento foi o tipo do aluno_id. O modelo às vezes envia o identificador como texto em vez de inteiro, e a versão inicial de comparar_materias_aluno só reconhecia inteiros, respondendo que o aluno não existia e derrubando silenciosamente os cenários de filtragem. A correção foi normalizar o identificador para aceitar inteiro, texto ou dicionário.

Também tivemos bastante dificuldade para lidar com o erro de thought signature do Gemini. O modelo exige que o campo thought_signature retorne na requisição seguinte a cada chamada de ferramenta, e o agentkit não preservava esse campo, interrompendo o fluxo. A solução foi preservar o campo nas duas direções da tradução, no módulo gemini_compat, e usar o modelo 3.8 no lugar do 3.5.

Outros limites observados: a abordagem por planejamento gera roteiros longos que podem esbarrar no limite de passos; o guard rail de intenção julga cada mensagem isoladamente, então recusas muito curtas e genéricas precisam ser ancoradas no tema de disciplinas; e as ferramentas dependem de uma API externa, cuja indisponibilidade afeta os testes.

## RELATÓRIO

### O que o agente faz

O agente é um orientador acadêmico do Bacharelado em TI: a partir do histórico do aluno e da oferta do semestre, ele monta uma grade curricular viável. O desenho segue a abordagem de planejamento (um plano é escrito antes da execução) somada a um laço de ferramentas: recebe_input_usuario coleta preferências e devolve propostas, comparar_materias_aluno cruza histórico com oferta e classifica em pode cursar, falta pré-requisito e já aprovado, consulta_turma_api e taxa_aprovacao_disciplina consultam a API de turmas do IMD, calcula_carga_hora_disciplinas soma a carga da grade, coleta_feedback registra a avaliação do aluno e encerra_planejamento finaliza a sessão. Um guard rail de intenção (classifica_intencao_usuario) protege o fluxo: pedidos fora do tema de matrícula são recusados antes de qualquer passo.

### Por que a tarefa é verificável

Cada decisão do agente deixa um rastro objetivo que o teste inspeciona sem depender de julgamento subjetivo do texto: as ações (a sequência de ferramentas chamadas), as observações (o texto que as ferramentas devolveram, como "Nenhuma disciplina encontrada", "falta pré-requisito" ou "JÁ APROVADO"), a saída (a grade final deve conter códigos de disciplina reais, e as regras de pré-requisito, duplicidade e teto de carga horária são checáveis contra os CSVs em dados) e o efeito colateral (coleta_feedback grava uma linha em dados/feedback.jsonl). As verificações olham padrões, e não strings exatas, para serem robustas às reformulações naturais do modelo.

### Principais decisões de projeto

A ideia de um agente orientador surgiu da alta demanda por orientação de matrícula e da alta aderência com a realidade dos discentes.

As regras são definidas previamente: o agente precisa conhecer uma quantidade mínima de informações do aluno para conseguir estruturar uma grade mínima.

O modelo 3.8 foi utilizado porque o 3.5 estava resultando em uma falha relacionada à thought signature. Esse foi o erro que mais nos deu trabalho: o Gemini exige que o campo thought_signature retorne na requisição seguinte a cada chamada de ferramenta, e o agentkit não preservava esse campo, interrompendo o fluxo. Contornamos preservando o campo nas duas direções da tradução (módulo gemini_compat) e trocando o modelo.

A quantidade de tokens foi um hiperparâmetro que precisou ser aumentado para o output ser exibido corretamente.

Os testes almejaram simular cenários reais de atendimento.

### Nota sobre ReAct e planejamento

Sobre o uso de ReAct ou planejamento para a tarefa que propusemos que o agente cumpra (sugestão de grades curriculares para discentes), convém ressaltar que inicialmente a abordagem utilizando o planejamento aparenta ser mais atrativa, tendo em vista que permite que o agente não realize uma sequência de perguntas ao estudante até o próprio usuário saber que ainda não possui uma determinada informação. Um exemplo real desse cenário consiste em relação ao usuário manifestar profundo interesse em relação a uma determinada disciplina, mas antes do agente orientador elaborar próximos passos, questionar ao aluno se ele cumpre os pré-requsisitos exigidos para cursá-la. Ou seja, o agente já possui um grande contexto prévio, pois conhece a sequência de etapas necessárias para cumprir a tarefa que recebeu. No ReAct, isso não necessariamente ocorre dado que "[...] ele decide o próximo passo a cada rodada, a partir do resultado do passo anterior".